# Clinical Notes — Exploratory Data Analysis
Analysis of 50 clinical notes for NER model training and guideline evaluation.

In [1]:
import json
import pandas as pd
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))
from src.config import CLINICAL_NOTES_PATH, GUIDELINES_PATH
print("Libraries loaded successfully")

Libraries loaded successfully


In [2]:
with open(CLINICAL_NOTES_PATH) as f:
    notes = json.load(f)
with open(GUIDELINES_PATH) as f:
    guidelines = json.load(f)
df = pd.DataFrame(notes)
print(f"Total notes: {len(df)}")
df.head()

Total notes: 50


,note_id,text
0,N001,45 year old male with fever and productive cou...
1,N002,"30 year old female presenting with headache, n..."
2,N003,60 year old male with chest pain radiating to ...
3,N004,25 year old female with dysuria and lower abdo...
4,N005,"50 year old male with cough, weight loss and n..."


## Dataset Statistics

In [3]:
df['text_length'] = df['text'].str.len()
df['word_count'] = df['text'].str.split().str.len()
print(f"Total notes: {len(df)}")
print(f"Avg length (chars): {df['text_length'].mean():.1f}")
print(f"Avg word count: {df['word_count'].mean():.1f}")
print(f"Min words: {df['word_count'].min()}")
print(f"Max words: {df['word_count'].max()}")
df[['text_length','word_count']].describe()

Total notes: 50
Avg length (chars): 93.4
Avg word count: 13.3
Min words: 10
Max words: 18


,text_length,word_count
count,50.000000,50.000000
mean,93.400000,13.340000
std,12.658787,1.825108
min,70.000000,10.000000
25%,84.250000,12.000000
50%,90.500000,13.000000
75%,102.750000,14.000000
max,122.000000,18.000000


## Entity Extraction Analysis

In [4]:
from src.preprocessing.text_processor import ClinicalAnnotator
annotator = ClinicalAnnotator()
records = []
for note in notes:
    result = annotator.annotate(note['note_id'], note['text'])
    ext = result['extracted']
    records.append({
        'note_id': note['note_id'],
        'age': ext['age'],
        'sex': ext['sex'],
        'diagnosis': ext['diagnosis'],
        'num_symptoms': len(ext['symptoms']),
        'num_medications': len(ext['medications']),
    })
ann_df = pd.DataFrame(records)
print("Extraction complete.")
ann_df.head(10)

Extraction complete.


,note_id,age,sex,diagnosis,num_symptoms,num_medications
0,N001,45 year old,male,pneumonia,2,1
1,N002,30 year old,female,meningitis,3,1
2,N003,60 year old,male,myocardial infarction,2,2
3,N004,25 year old,female,urinary tract infection,2,1
4,N005,50 year old,male,tuberculosis,3,2
5,N006,70 year old,female,asthma exacerbation,2,1
6,N007,5 year old,male,gastroenteritis,3,1
7,N008,35 year old,female,tonsillitis,2,1
8,N009,55 year old,male,diabetes mellitus,2,1
9,N010,40 year old,female,migraine,2,1


## Diagnosis Distribution

In [5]:
print("Diagnosis value counts:")
print(ann_df['diagnosis'].value_counts())
print(f"\nUnique diagnoses: {ann_df['diagnosis'].nunique()}")
print(f"Missing diagnosis: {ann_df['diagnosis'].isna().sum()}")

Diagnosis value counts:
diagnosis
pneumonia                  7
meningitis                 5
myocardial infarction      5
urinary tract infection    5
tonsillitis                5
gastroenteritis            5
diabetes mellitus          5
tuberculosis               4
migraine                   4
asthma exacerbation        1
Name: count, dtype: int64

Unique diagnoses: 10
Missing diagnosis: 4


## Sex and Age Distribution

In [6]:
print("Sex distribution:")
print(ann_df['sex'].value_counts())
print(f"\nMissing age: {ann_df['age'].isna().sum()}")
print(f"Missing sex: {ann_df['sex'].isna().sum()}")

Sex distribution:
sex
male      26
female    24
Name: count, dtype: int64

Missing age: 0
Missing sex: 0


## Guideline Coverage

In [7]:
print(f"Total guideline conditions: {len(guidelines)}")
diagnosed = set(ann_df['diagnosis'].dropna().str.lower())
covered = diagnosed.intersection(set(guidelines.keys()))
print(f"Unique diagnoses in notes: {len(diagnosed)}")
print(f"Covered by guidelines: {len(covered)}")
print(f"\nCovered conditions: {sorted(covered)}")
not_covered = diagnosed - set(guidelines.keys())
print(f"\nNot covered: {not_covered if not_covered else 'None — 100% coverage'}")

Total guideline conditions: 19
Unique diagnoses in notes: 10
Covered by guidelines: 10

Covered conditions: ['asthma exacerbation', 'diabetes mellitus', 'gastroenteritis', 'meningitis', 'migraine', 'myocardial infarction', 'pneumonia', 'tonsillitis', 'tuberculosis', 'urinary tract infection']

Not covered: None — 100% coverage


## Data Quality Summary

In [8]:
print("=== Data Quality Report ===")
print(f"Total notes: {len(ann_df)}")
print(f"Notes with complete entities: {ann_df.dropna(subset=['age','sex','diagnosis']).shape[0]}")
print(f"Notes missing diagnosis: {ann_df['diagnosis'].isna().sum()}")
print(f"Notes with multiple medications: {(ann_df['num_medications'] > 1).sum()}")
print(f"Average medications per note: {ann_df['num_medications'].mean():.2f}")
print(f"Average symptoms per note: {ann_df['num_symptoms'].mean():.2f}")
print("\n=== Key Findings ===")
print("- 50 synthetic clinical notes")
print("- 10 distinct diagnoses")
print("- 100% guideline coverage")
print("- Short structured free-text suitable for NER")
print("- No duplicate notes detected")
print("- Transfer learning approach chosen due to small dataset size")

=== Data Quality Report ===
Total notes: 50
Notes with complete entities: 46
Notes missing diagnosis: 4
Notes with multiple medications: 5
Average medications per note: 1.10
Average symptoms per note: 1.78

=== Key Findings ===
- 50 synthetic clinical notes
- 10 distinct diagnoses
- 100% guideline coverage
- Short structured free-text suitable for NER
- No duplicate notes detected
- Transfer learning approach chosen due to small dataset size
